In [ ]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep
import os

from myutils import email_notify


from datetime import datetime
today = datetime.strftime(datetime.today(), '%Y%m%d')
del datetime

In [ ]:
today

In [ ]:
# with RaspiLED() as led:
#     led.check()

In [ ]:
np.linspace(0.1, 0.4, 61)

In [ ]:
sampling_rate = 20 #Hz
freq_list = np.linspace(0.1, 0.4, 61)

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int) #us

A = 5
samples = 50
repeat = 200

interval = 50e-3 #s, 50 ms
exposure_time = 500e-6 #s, 500 us
timeout_milisec = 3000 #ms, 3000 ms

measurement = 'DI'

In [ ]:
############
# NO NOISE #
############

# @email_notify('hcnzj@qq.com')
def main():
    with EasyDcam() as dcam, EasyALP4() as alp:
        for i, picture_time in enumerate(pic_time):
            print(f'({i + 1}): Current sensor temperature is {dcam.ez_temperature()}')
            if dcam.ez_temperature() >= -30:
                raise RuntimeError("qCMOS's temperature is too high.")

            ground_truth = np.round(freq_list[i], 5)

            alp.ez_load_seq([alp.ez_single_pixel(0), alp.ez_single_pixel(A)], picture_time)
            dcam.ez_exposure_time(exposure_time)
            dcam.ez_triggersource_masterpluse(samples, interval)

            if measurement.upper() == 'SPADE':
                dcam.ez_roi(**SPADE.ROI)
            elif measurement.upper() == 'DI':
                dcam.ez_roi(**DI.ROI)

            raw, timestamp = [], []
            for _ in tqdm(range(repeat)):
                dcam.buf_alloc(samples)
                dcam.cap_snapshot()

                alp.Run()
                sleep(1e-6)
                dcam.cap_firetrigger()

                dcam.ez_wait_capture(timeout_milisec)

                dcam.cap_stop()
                alp.Halt()

                raw_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = dcam.ez_read_buf(frame)
                    raw_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                dcam.buf_release()

                raw.append(raw_)
                timestamp.append(timestamp_)

            raw = np.array(raw)
            timestamp = np.array(timestamp)

            if not os.path.exists(f'__raw__/{today}'):
                os.makedirs(f'__raw__/{today}')
            if not os.path.exists(f'__estimates__/{today}'):
                os.makedirs(f'__estimates__/{today}')

            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_raw.npy', raw)
            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_timestamp.npy', timestamp)

            metadata = MetaData(measurement, ground_truth, A*DMD.PIXEL_SIZE/2, timestamp)
            est = FrequencyEstimation.FromRaw(np.array(raw), metadata)
            est.savez(f'./__estimates__/{today}/{measurement.lower()}_{ground_truth}.npz')

            print(f'({i + 1}): {dcam.lasterr()}')


if __name__ == '__main__':
    main()

In [ ]:
raise RuntimeError('STOP HERE')